# Mission 5: Inside a Tiny AI 🧠

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/notebooks/05_inside_a_tiny_ai.ipynb)

> **From your syllabus**
> - *ML-I*, Unit IV (feed-forward networks, backpropagation) and *Convex Optimization*, Unit I (chain rule)
> - *Linear Algebra*: matrix products, and softmax from Mission 2

**Your mission:** open the box. A neural network is a composition of functions, and backpropagation is just the chain rule you already know. Then we reach the idea behind modern chatbots: **attention**, which is `softmax(QKᵀ)V`.

> 📊 **Progress tracking: please read.** The next cell draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its full code is in the cell, so you can read it. Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (run me first; the code is short, read it)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 5

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"0": {"title": "Ignite", "checkpoints": ["m00-ready", "m00-done"]}, "1": {"title": "Golf", "checkpoints": ["m01-grad", "m01-done"]}, "2": {"title": "Digit Reader", "checkpoints": ["m02-grad", "m02-done"]}, "3": {"title": "Selfie", "checkpoints": ["m03-ey", "m03-pca", "m03-done"]}, "4": {"title": "Mini-Google", "checkpoints": ["m04-markov", "m04-done"]}, "5": {"title": "Tiny AI", "checkpoints": ["m05-backprop", "m05-done"]}}
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, n) for n in range(len(MISSIONS))]
    for n, (st, got, tot) in enumerate(stats):
        x = 45 + n * 108
        title = MISSIONS[str(n)]["title"]
        if n:
            ok = stats[n - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">M{n} {title}</text>')
    finished = [n for n, t in enumerate(stats) if t[0] == "done"]
    todo = [f"M{n}" for n, t in enumerate(stats) if t[0] == "todo"]
    line = f"<b>{len(finished)} of {len(stats)} missions finished</b> &middot; now in <b>Mission {MISSION}</b> ({stats[MISSION][1]}/{stats[MISSION][2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else " &middot; every mission visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


_start()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

np.random.seed(0)
X, t = make_moons(n_samples=300, noise=0.2, random_state=0)   # two interleaved half-moons
plt.scatter(X[:, 0], X[:, 1], c=t, cmap="coolwarm", s=15); plt.title("a problem no straight line can solve"); plt.show()

## 1. The network
One hidden layer with 8 units:
$$h=\tanh(XW_1+b_1),\qquad p=\sigma(hW_2+b_2).$$
A logistic regression (Mission 2) is the second half; the hidden layer learns new features first.

**Core 1:** fill in the forward pass.

<details><summary>Hint</summary>

`z1 = X @ W1 + b1`, then `h = np.tanh(z1)`. Next `z2 = h @ W2 + b2` and `p = 1 / (1 + np.exp(-z2))`.
</details>

In [ ]:
def init(hidden=8, seed=0):
    r = np.random.RandomState(seed)
    return {"W1": r.randn(2, hidden) * 0.5, "b1": np.zeros(hidden),
            "W2": r.randn(hidden, 1) * 0.5, "b2": np.zeros(1)}

def forward(P, X):
    z1 = ____                      # TODO: X @ W1 + b1
    h = ____                       # TODO: tanh of z1
    z2 = ____                      # TODO: h @ W2 + b2
    p = ____                       # TODO: sigmoid of z2
    return h, p

def loss(P, X, t):
    _, p = forward(P, X)
    p, tt = p.ravel(), t
    return -np.mean(tt * np.log(p + 1e-12) + (1 - tt) * np.log(1 - p + 1e-12))

P = init()
print("initial loss (should be near 0.69):", round(loss(P, X, t), 3))

## 2. Backpropagation = the chain rule, right to left
Let $n$ be the number of examples. From Mission 2 the output layer gives $\delta_2=\partial L/\partial z_2=(p-t)/n$. Then, layer by layer:

| step | formula |
|---|---|
| $\partial L/\partial W_2$ | $h^\top\delta_2$ |
| $\partial L/\partial b_2$ | $\sum\delta_2$ |
| $\partial L/\partial h$ | $\delta_2W_2^\top$ |
| $\delta_1=\partial L/\partial z_1$ | $(\partial L/\partial h)\odot(1-h^2)$ since $\tanh'=1-\tanh^2$ |
| $\partial L/\partial W_1$ | $X^\top\delta_1$ |
| $\partial L/\partial b_1$ | $\sum\delta_1$ |

**Core 2:** fill in the four blanks and let the finite-difference check judge your calculus.

In [ ]:
def backward(P, X, t):
    h, p = forward(P, X)
    n = len(t)
    d2 = (p - t.reshape(-1, 1)) / n
    dW2 = ____                     # TODO: h.T @ d2
    db2 = d2.sum(axis=0)
    dh = ____                      # TODO: d2 @ W2.T
    d1 = ____                      # TODO: dh * (1 - h**2)
    dW1 = ____                     # TODO: X.T @ d1
    db1 = d1.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

def check(P, X, t, h=1e-5):
    G = backward(P, X, t)
    for name in P:
        num = np.zeros_like(P[name])
        for idx in np.ndindex(*P[name].shape):
            old = P[name][idx]
            P[name][idx] = old + h; lp = loss(P, X, t)
            P[name][idx] = old - h; lm = loss(P, X, t)
            P[name][idx] = old
            num[idx] = (lp - lm) / (2 * h)
        assert np.allclose(G[name], num, atol=1e-6), f"gradient for {name} is wrong"
    print("All gradients correct ✔  (backprop = chain rule)")

check(init(), X, t)
checkpoint("m05-backprop")

## 3. Train it
Same golf swing as Missions 1 and 2, now with a few more parameters.

In [ ]:
P = init()
history = []
for step in range(2000):
    G = backward(P, X, t)
    for name in P:
        P[name] -= 1.0 * G[name]
    history.append(loss(P, X, t))

_, p = forward(P, X)
print("training accuracy:", np.mean((p.ravel() > 0.5) == t))
plt.plot(history); plt.xlabel("stroke"); plt.ylabel("loss"); plt.show()

gx, gy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
_, pg = forward(P, np.c_[gx.ravel(), gy.ravel()])
plt.contourf(gx, gy, pg.reshape(gx.shape), levels=20, cmap="coolwarm", alpha=.6)
plt.scatter(X[:, 0], X[:, 1], c=t, cmap="coolwarm", s=15, edgecolors="k", linewidths=.3)
plt.title("the hidden layer bends the boundary"); plt.show()

## 4. From networks to attention
Chatbots read a sentence as a list of vectors (one per word) and let every word **look at** every other word. Each token $i$ gets three vectors from learned matrices: a **query** $q_i=x_iW_Q$ ("what am I looking for?"), a **key** $k_i=x_iW_K$ ("what do I offer?") and a **value** $v_i=x_iW_V$ ("what I will pass on"). Then
$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt d}\right)V .$$
Every row of the softmax sums to 1 (Mission 2), so each output is a **weighted average of values**; the weights say who is paying attention to whom.

**Core 3:** fill in the attention formula.

<details><summary>Hint</summary>

`scores = Q @ K.T / np.sqrt(d)`, `weights = softmax(scores)`, `out = weights @ V`.
</details>

In [ ]:
tokens = ["the", "cat", "sat", "down"]
d = 8
r = np.random.RandomState(1)
E = r.randn(len(tokens), d)                              # one random embedding per word
WQ, WK, WV = [r.randn(d, d) / np.sqrt(d) for _ in range(3)]   # scaled so q, k entries have variance about 1
Q, K, V = E @ WQ, E @ WK, E @ WV

def softmax_rows(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)

def attention(Q, K, V):
    scores = ____                 # TODO: Q @ K.T / sqrt(d)
    weights = ____                # TODO: softmax_rows(scores)
    out = ____                    # TODO: weights @ V
    return weights, out

weights, out = attention(Q, K, V)
print("each row sums to 1:", np.allclose(weights.sum(axis=1), 1))
plt.imshow(weights, cmap="Blues"); plt.xticks(range(4), tokens); plt.yticks(range(4), tokens)
plt.xlabel("looked at"); plt.ylabel("looking"); plt.colorbar(); plt.title("attention weights"); plt.show()

> **Why divide by $\sqrt d$?** If entries of $q,k$ are independent with variance 1, then $q\cdot k$ has variance $d$. Dividing by $\sqrt d$ keeps the scores at variance 1 so the softmax is not saturated (Applied Probability Unit II: variance of a sum). Check it below.

In [ ]:
big = np.random.randn(20000, 64), np.random.randn(20000, 64)
dots = (big[0] * big[1]).sum(axis=1)
print("variance of q.k          :", round(dots.var(), 1), "(about d = 64)")
print("variance of q.k / sqrt(d):", round((dots / 8).var(), 2), "(about 1)")

## Stretch: do not peek at the future
A language model predicting the next word must not look ahead. Add a **causal mask**: set the scores above the diagonal to $-\infty$ before the softmax. Fill in and check that the weight matrix is lower-triangular.

<details><summary>Hint</summary>

`np.triu(np.ones((4, 4)), k=1) == 1` marks the future positions; use `np.where(mask, -1e9, scores)`.
</details>

In [ ]:
scores = Q @ K.T / np.sqrt(d)
mask = np.triu(np.ones((4, 4)), k=1) == 1
masked_scores = ____              # TODO: replace future positions by -1e9
w_causal = softmax_rows(masked_scores)
print(np.round(w_causal, 2))
print("lower-triangular:", np.allclose(np.triu(w_causal, k=1), 0))

In [ ]:
checkpoint("m05-done")                      # Mission 5 complete: watch your map update at the top

## Quest (take-home)
Add a second hidden layer to the network (write `backward` for it using the same table), or train the network on the digits from Mission 2 and compare with softmax regression. Post your accuracy and what changed.

## You just used...
- **Backpropagation** as the chain rule on a computation graph (ML-I Unit IV, Convex Optimization Unit I)
- **tanh and sigmoid** non-linearities that let a network bend decision boundaries
- **softmax** as weights, and $\mathrm{softmax}(QK^\top/\sqrt d)V$, the heart of Transformers (the same family as the models behind modern chatbots)
- the **variance of a sum** to justify the $\sqrt d$ scaling

## Your CV line
> *Built a neural network with backpropagation from scratch in NumPy (gradients verified numerically) and implemented scaled dot-product attention with causal masking.*

Finished all six? Head to the showcase, then fill in the `portfolio-template/` folder in the repo.